In [42]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()
spark =SparkSession. \
        builder. \
        config('spark.ui.port','0'). \
        config('spark.shuffle.useOldFetchProtocol','true'). \
        config('spark.sql.warehouse.dir',f"/user/{username}/warehouse"). \
        enableHiveSupport(). \
        master('yarn'). \
        getOrCreate()

In [43]:
loans_repay_raw_df = spark.read \
.format("csv") \
.option("inferSchema",True) \
.option("header",True) \
.load("/user/itv026162/lendingclub/project/raw/loans_repayments_data_csv")

In [44]:
loans_repay_raw_df

loan_id,total_rec_prncp,total_rec_int,total_rec_late_fee,total_pymnt,last_pymnt_amnt,last_pymnt_d,next_pymnt_d
491699,7000.0,1444.94,0.0,8444.942493933,954.65,Dec-2012,null
491685,15000.0,2583.58,0.0,17583.584829786698,517.23,Mar-2013,null
491667,6400.0,807.51,0.0,7207.5090884128,209.97,Mar-2013,null
491160,4000.0,963.31,0.0,4963.3064239472,144.87,Mar-2013,null
491675,20000.0,2887.0,0.0,22886.9924287214,16.27,Jul-2011,null
491668,1773.57,591.33,0.0,2957.37,394.94,Mar-2011,null
491663,5500.0,235.4,0.0,5735.4,2147.02,Sep-2010,null
491632,10000.0,1466.31,0.0,11466.3124714841,7274.73,Apr-2011,null
491618,24999.99,2085.75,0.0,27085.737419131998,21866.75,Oct-2010,null
491622,25000.0,13079.81,0.0,38079.8091525031,7912.97,Feb-2015,null


In [45]:
loans_repay_raw_df.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- total_rec_prncp: string (nullable = true)
 |-- total_rec_int: string (nullable = true)
 |-- total_rec_late_fee: string (nullable = true)
 |-- total_pymnt: string (nullable = true)
 |-- last_pymnt_amnt: string (nullable = true)
 |-- last_pymnt_d: string (nullable = true)
 |-- next_pymnt_d: string (nullable = true)



In [46]:
loan_repay_schema = 'loan_id string,total_principal_received float,total_interest_received float,total_late_fee_received float,total_payment_received float,last_payment_amount float,last_payment_date string,next_payment_date string'

In [47]:
loans_repay_raw_df = spark.read \
.format("csv") \
.option("header", True) \
.schema(loan_repay_schema) \
.load("/user/itv026162/lendingclub/project/raw/loans_repayments_data_csv")

In [48]:
loans_repay_raw_df

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date
491699,7000.0,1444.94,0.0,8444.942,954.65,Dec-2012,null
491685,15000.0,2583.58,0.0,17583.584,517.23,Mar-2013,null
491667,6400.0,807.51,0.0,7207.5093,209.97,Mar-2013,null
491160,4000.0,963.31,0.0,4963.3066,144.87,Mar-2013,null
491675,20000.0,2887.0,0.0,22886.992,16.27,Jul-2011,null
491668,1773.57,591.33,0.0,2957.37,394.94,Mar-2011,null
491663,5500.0,235.4,0.0,5735.4,2147.02,Sep-2010,null
491632,10000.0,1466.31,0.0,11466.3125,7274.73,Apr-2011,null
491618,24999.99,2085.75,0.0,27085.738,21866.75,Oct-2010,null
491622,25000.0,13079.81,0.0,38079.81,7912.97,Feb-2015,null


In [49]:
loans_repay_raw_df.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- total_principal_received: float (nullable = true)
 |-- total_interest_received: float (nullable = true)
 |-- total_late_fee_received: float (nullable = true)
 |-- total_payment_received: float (nullable = true)
 |-- last_payment_amount: float (nullable = true)
 |-- last_payment_date: string (nullable = true)
 |-- next_payment_date: string (nullable = true)



In [50]:
from pyspark.sql.functions import current_timestamp
loans_repay_df_ingested = loans_repay_raw_df.withColumn("ingest_date",current_timestamp())

In [51]:
loans_repay_df_ingested

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,ingest_date
491699,7000.0,1444.94,0.0,8444.942,954.65,Dec-2012,null,2026-11-18 16:17:...
491685,15000.0,2583.58,0.0,17583.584,517.23,Mar-2013,null,2026-11-18 16:17:...
491667,6400.0,807.51,0.0,7207.5093,209.97,Mar-2013,null,2026-11-18 16:17:...
491160,4000.0,963.31,0.0,4963.3066,144.87,Mar-2013,null,2026-11-18 16:17:...
491675,20000.0,2887.0,0.0,22886.992,16.27,Jul-2011,null,2026-11-18 16:17:...
491668,1773.57,591.33,0.0,2957.37,394.94,Mar-2011,null,2026-11-18 16:17:...
491663,5500.0,235.4,0.0,5735.4,2147.02,Sep-2010,null,2026-11-18 16:17:...
491632,10000.0,1466.31,0.0,11466.3125,7274.73,Apr-2011,null,2026-11-18 16:17:...
491618,24999.99,2085.75,0.0,27085.738,21866.75,Oct-2010,null,2026-11-18 16:17:...
491622,25000.0,13079.81,0.0,38079.81,7912.97,Feb-2015,null,2026-11-18 16:17:...


In [52]:
loans_repay_df_ingested.createOrReplaceTempView("loan_repayment")

In [53]:
loans_repay_df_ingested.count()

2260701

In [54]:
spark.sql("select count(*) from loan_repayment where total_principal_received is null")

count(1)
69


In [55]:
columns_to_check =["total_principal_received","total_interest_received","total_late_fee_received","total_payment_received","last_payment_amount"]

In [56]:
loans_repay_filtered_df = loans_repay_df_ingested.na.drop(subset=columns_to_check)

In [57]:
loans_repay_filtered_df.count()

2260498

In [58]:
2260701 - 2260498

203

In [59]:
loans_repay_filtered_df.createOrReplaceTempView("loans_repayments")

In [60]:
spark.sql("select count(*) from loans_repayments where total_payment_received = 0.0")

count(1)
995


In [61]:
spark.sql("select count(*) from loans_repayments where total_payment_received = 0.0 and total_principal_received != 0.0")

count(1)
46


In [62]:
spark.sql("select * from loans_repayments where total_payment_received = 0.0 and total_principal_received != 0.0")

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,ingest_date
485818,14640.096,13388.84,13000.0,0.0,0.0,0.0,Mar-2013,2026-11-18 16:17:...
485471,29620.818,29134.64,25000.0,0.0,0.0,0.0,Mar-2013,2026-11-18 16:17:...
482256,8735.611,7479.87,8000.0,0.0,0.0,0.0,Feb-2011,2026-11-18 16:17:...
478160,410.0,407.36,0.0,0.0,143.1,410.0,null,2026-11-18 16:17:...
476557,28865.18,24164.67,5692.31,0.0,6972.59,19916.78,Dec-2010,2026-11-18 16:17:...
472516,25951.482,24731.76,25000.0,0.0,0.0,0.0,May-2010,2026-11-18 16:17:...
472197,12048.13,12018.01,10000.0,0.0,0.0,0.0,Jan-2013,2026-11-18 16:17:...
467364,29216.791,29066.19,24250.0,0.0,0.0,0.0,Dec-2012,2026-11-18 16:17:...
399499,26557.729,26336.41,24000.0,0.0,0.0,0.0,Dec-2010,2026-11-18 16:17:...
451482,7587.5513,7587.55,7000.0,0.0,0.0,0.0,Jan-2011,2026-11-18 16:17:...


In [63]:
from pyspark.sql.functions import when, col, length

In [64]:
loans_payments_fixed_df = loans_repay_filtered_df.withColumn("total_payment_received",
                                   when(
                                   (col("total_principal_received") != 0.0 ) &
                                   (col("total_payment_received") == 0.0),
                                   col("total_principal_received") +  col("total_interest_received") + col("total_late_fee_received")  
                                ).otherwise(col("total_payment_received"))
                                  )

In [65]:
loans_payments_fixed_df.filter("loan_id == '1064185'")

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,ingest_date
1064185,11600.98,11600.98,10000.0,33201.96,0.0,0.0,Dec-2014,2026-11-18 16:17:...


In [66]:
loans_payments_fixed2_df = loans_payments_fixed_df.filter("total_payment_received != 0.0")

In [67]:
loans_payments_fixed2_df

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,ingest_date
491699,7000.0,1444.94,0.0,8444.942,954.65,Dec-2012,null,2026-11-18 16:17:...
491685,15000.0,2583.58,0.0,17583.584,517.23,Mar-2013,null,2026-11-18 16:17:...
491667,6400.0,807.51,0.0,7207.5093,209.97,Mar-2013,null,2026-11-18 16:17:...
491160,4000.0,963.31,0.0,4963.3066,144.87,Mar-2013,null,2026-11-18 16:17:...
491675,20000.0,2887.0,0.0,22886.992,16.27,Jul-2011,null,2026-11-18 16:17:...
491668,1773.57,591.33,0.0,2957.37,394.94,Mar-2011,null,2026-11-18 16:17:...
491663,5500.0,235.4,0.0,5735.4,2147.02,Sep-2010,null,2026-11-18 16:17:...
491632,10000.0,1466.31,0.0,11466.3125,7274.73,Apr-2011,null,2026-11-18 16:17:...
491618,24999.99,2085.75,0.0,27085.738,21866.75,Oct-2010,null,2026-11-18 16:17:...
491622,25000.0,13079.81,0.0,38079.81,7912.97,Feb-2015,null,2026-11-18 16:17:...


In [68]:
loans_payments_fixed2_df.filter("last_payment_date = 0.0").count()

48

In [69]:
loans_payments_fixed2_df.filter("next_payment_date = 0.0").count()

24

In [70]:
loans_payments_ldate_fixed_df = loans_payments_fixed2_df.withColumn(
                                    "last_payment_date",
                                    when(
                                    (col("last_payment_date") == 0.0),
                                    None
                                    ).otherwise(col("last_payment_date"))
                                )

In [71]:
loans_payments_ndate_fixed_df = loans_payments_ldate_fixed_df.withColumn(
                                    "next_payment_date",
                                    when(
                                    (col("next_payment_date") == 0.0),
                                    None
                                    ).otherwise(col("next_payment_date"))
                                )

In [72]:
loans_payments_ndate_fixed_df.filter("last_payment_date = 0.0").count()

0

In [ ]:
loans_payments_ndate_fixed_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path","/user/itv026162/lendingclub/project/cleaned/loan_repay_data_csv") \
.save()

In [41]:
spark.stop()